
# Tile2Net aerial pedestrian-infrastructure pipeline (inference-only)

Runs the full Tile2Net pipeline on a **Colab GPU runtime** and produces the
router-contract GeoJSON files for the accessibility-aware walking/biking
project (DC area).

**Before running:** `Runtime -> Change runtime type -> GPU (T4/L4/A100)`.

Pipeline: env check -> install tile2net (pinned commit) -> Boston example
(environment validation) -> DC test area -> DC demo area -> schema
inspection -> contract GeoJSON export -> preview maps -> OSM comparison
metrics -> payload zip download.

**Drop the payload into the repo:**
| payload path | repo path |
|---|---|
| `data/*.geojson` | `data/` |
| `ml_aerial_previews/*` | `ml/aerial/previews/` |
| `metrics/aerial_vs_osm.json` | `metrics/` |
| `parquet_debug/*` | anywhere (debugging only, not committed) |

Notes:
* `DEMO_BBOX` placeholder: ~1 km2 Logan Circle, DC. Replace in the config
  cell when the real demo bbox is set (format `[lat, lon, lat, lon]`).
* DC and VA imagery sources are supported; **Maryland is not** -- do not run
  demo areas there.
* Runtime on T4: ~15-25 min total, < 2 GB disk.


In [ ]:

import os
import subprocess
import torch

print("nvidia-smi:")
subprocess.run(["nvidia-smi"], check=False)
cuda_ok = torch.cuda.is_available()
print("torch.cuda.is_available():", cuda_ok, "| torch:", torch.__version__)
assert cuda_ok, (
    "No CUDA GPU. Tile2Net inference requires one. "
    "Select Runtime > Change runtime type > GPU (T4/L4/A100) and rerun."
)



## Config — edit `DEMO_BBOX` here


In [ ]:

import os

BASE = "/kaggle/working" if os.path.exists("/kaggle") else "/content"

# [lat, lon, lat, lon] — top-left then bottom-right
DEMO_BBOX = (38.9182, -77.0380, 38.9092, -77.0270)   # ~1 km2 Logan Circle, DC — REPLACE when real bbox exists
TEST_BBOX = (38.9162, -77.0340, 38.9142, -77.0310)   # small fast-validation box
DEMO_REGION = "washington-dc"  # for in_training_cities: washington-dc | boston | cambridge | nyc

TILE2NET_URL = "https://github.com/VIDA-NYU/tile2net.git"
TILE2NET_COMMIT = "c737a4e"  # pinned for reproducibility; keep in sync with SETUP.md

OUTPUT_ROOT = f"{BASE}/output"
DATA_DIR = f"{BASE}/data"
PREVIEW_DIR = f"{BASE}/ml_aerial_previews"
METRICS_DIR = f"{BASE}/metrics"
PQ_DIR = f"{BASE}/parquet_debug"

for d in (OUTPUT_ROOT, DATA_DIR, PREVIEW_DIR, METRICS_DIR, PQ_DIR):
    os.makedirs(d, exist_ok=True)

print("BASE =", BASE)



## Install tile2net (pinned)


In [ ]:

import subprocess

repo_dir = f"{BASE}/tile2net"
if not os.path.exists(f"{repo_dir}/.git"):
    subprocess.run(["git", "clone", TILE2NET_URL, repo_dir], check=True)
subprocess.run(["git", "-C", repo_dir, "checkout", TILE2NET_COMMIT], check=True)
subprocess.run(["git", "-C", repo_dir, "rev-parse", "--short", "HEAD"], check=True)


In [ ]:

import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir], check=True)
print("tile2net installed")



## Pipeline helpers


In [ ]:

import json
import re
import subprocess as sp
import sys


def bbox_str(bb):
    return ",".join(str(v) for v in bb)


def generate(location, name, out_root, dump_percent=0):
    cmd = [
        sys.executable, "-m", "tile2net", "generate",
        "--location", location, "--name", name,
        "--output", out_root, "--dump_percent", str(dump_percent),
    ]
    p = sp.run(cmd, cwd=repo_dir, capture_output=True, text=True)
    print(p.stdout[-3000:])
    print(p.stderr[-3000:])
    m = re.search(r"INFO\s+Dumping to (\S+\.json)", p.stdout + "\n" + p.stderr)
    if not m:
        raise RuntimeError(f"generate failed for {name}; see logs above")
    city_info = m.group(1)
    print("city_info:", city_info)
    return city_info


def infer(city_info, dump_percent=0, eval_mode="test"):
    cmd = [
        sys.executable, "-m", "tile2net", "inference",
        "--local", "--eval", eval_mode,
        "--city_info", city_info,
        "--dump_percent", str(dump_percent),
    ]
    p = sp.run(cmd, cwd=repo_dir, capture_output=True, text=True)
    print(p.stdout[-6000:])
    if p.returncode != 0:
        print(p.stderr[-6000:])
        raise RuntimeError("inference failed; see stderr above")
    print("inference ok:", city_info)
    return city_info



## 1. Environment validation — Boston example (headless `examples/example.sh`)


In [ ]:

BOSTON_BBOX = "42.35555189953313,-71.07168915322092,42.35364837213307,-71.06437423368418"

info = generate(BOSTON_BBOX, "example", OUTPUT_ROOT, dump_percent=100)
infer(info, dump_percent=100)
print("Boston example done — environment validated")



## 2. DC test area (validation, small)


In [ ]:

info_test = generate(bbox_str(TEST_BBOX), "test_area", OUTPUT_ROOT, dump_percent=0)
infer(info_test, dump_percent=0)



## 3. DC demo area


In [ ]:

info_demo = generate(bbox_str(DEMO_BBOX), "demo", OUTPUT_ROOT, dump_percent=0)
infer(info_demo, dump_percent=0)



## 4. Inspect raw outputs (names + schemas — mirror of SETUP.md)


In [ ]:

import geopandas as gpd
import glob

for area, name in (("test_area", "test_area"), ("demo", "demo")):
    net = f"{OUTPUT_ROOT}/{area}/network/{name}-network.parquet"
    pol = f"{OUTPUT_ROOT}/{area}/polygons/{name}-polygons.parquet"
    print(f"--- {area} ---")
    if os.path.exists(net):
        g = gpd.read_parquet(net)
        print("network:", len(g), "| crs:", g.crs, "| geom:", g.geom_type.value_counts().to_dict())
        print("network f_type:", g["f_type"].value_counts().to_dict())
    if os.path.exists(pol):
        g = gpd.read_parquet(pol)
        print("polygons:", len(g), "| crs:", g.crs, "| geom:", g.geom_type.value_counts().to_dict())
        print("polygons f_type:", g["f_type"].value_counts().to_dict())

print("--- all output dirs ---")
for d in sorted(glob.glob(f"{OUTPUT_ROOT}/**/", recursive=True))[:60]:
    print(d)



## 5. Run the postprocessing scripts (embedded copies of `ml/aerial/`)

The three `ml/aerial/` scripts are embedded below (base64, injected from the
committed files at build time) so the notebook is self-contained — no GitHub
dependency. If you edit those scripts, rebuild the notebook or re-paste.


In [ ]:

import base64
import pathlib

script_b64 = {
    "convert_tile2net_output.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJDb252ZXJ0IHRpbGUybmV0IHZlY3RvciBvdXRwdXRzIHRvIHRoZSByb3V0ZXIgY29udHJhY3QgR2VvSlNPTi4KClJlYWRzIHRoZSBHZW9QYXJxdWV0IGZpbGVzIHRpbGUybmV0J3MgaW5mZXJlbmNlIHN0ZXAgd3JpdGVzOgogIDxwcm9qZWN0LWRpcj4vbmV0d29yay88cHJvamVjdC1uYW1lPi1uZXR3b3JrLnBhcnF1ZXQgICAoTGluZVN0cmluZ3MsIEVQU0c6NDMyNikKICA8cHJvamVjdC1kaXI+L3BvbHlnb25zLzxwcm9qZWN0LW5hbWU+LXBvbHlnb25zLnBhcnF1ZXQgKFBvbHlnb25zLCBFUFNHOjQzMjYpCgpXcml0ZXMgTGluZVN0cmluZyAvIFBvbHlnb24gR2VvSlNPTiBpbiBFUFNHOjQzMjYgd2l0aCBwcm9wZXJ0aWVzCiAgeyJzb3VyY2UiOiAibWwiLCAia2luZCI6ICJzaWRld2FsayJ8ImNyb3Nzd2FsayJ8ImZvb3RwYXRoInwiY29ubmVjdG9yIiwKICAgImNvbmZpZGVuY2UiOiBudWxsfQoKdGlsZTJuZXQgZXhwb3NlcyBubyBwZXItZmVhdHVyZSBzb2Z0bWF4L2NvbmZpZGVuY2UgaW4gaXRzIHZlY3RvciBvdXRwdXRzLCBzbwpjb25maWRlbmNlIGlzIGFsd2F5cyBudWxsIChkb2N1bWVudGVkIGluIFNFVFVQLm1kOyBuZXZlciBpbnZlbnRlZCkuCgpFeGNsdWRlZCBmX3R5cGUgdmFsdWVzIChub3QgZGV0ZWN0ZWQgd2Fsa2FibGUgZmVhdHVyZXMpOgogIHBvbHlnb25zOiByb2FkLCBtZWRpYW5zCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHBhdGhsaWIKaW1wb3J0IHN5cwoKaW1wb3J0IGdlb3BhbmRhcyBhcyBncGQKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBzaGFwZWx5CgpMSU5FX0tJTkRTID0gewogICAgInNpZGV3YWxrIjogInNpZGV3YWxrIiwKICAgICJjcm9zc3dhbGsiOiAiY3Jvc3N3YWxrIiwKICAgICJmb290cGF0aCI6ICJmb290cGF0aCIsCiAgICAjIHRpbGUybmV0LWdlbmVyYXRlZCBzdGl0Y2hpbmcgbGluZXMg4oCUIGtlcHQgYXMga2luZCAiY29ubmVjdG9yIgogICAgInNpZGV3YWxrX2Nvbm5lY3Rpb24iOiAiY29ubmVjdG9yIiwKfQpQT0xZR09OX0tJTkRTID0gewogICAgInNpZGV3YWxrIjogInNpZGV3YWxrIiwKICAgICJjcm9zc3dhbGsiOiAiY3Jvc3N3YWxrIiwKICAgICJmb290cGF0aCI6ICJmb290cGF0aCIsCn0KCgpkZWYgX3RvXzQzMjYoZ2RmOiBncGQuR2VvRGF0YUZyYW1lKSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgaWYgZ2RmLmNycyBpcyBOb25lOgogICAgICAgIHJldHVybiBnZGYuc2V0X2Nycyg0MzI2KQogICAgaWYgZ2RmLmNycy50b19lcHNnKCkgIT0gNDMyNjoKICAgICAgICByZXR1cm4gZ2RmLnRvX2Nycyg0MzI2KQogICAgcmV0dXJuIGdkZgoKCmRlZiBfbWFwX2tpbmRzKGdkZjogZ3BkLkdlb0RhdGFGcmFtZSwgbGF5ZXI6IHN0ciwga2luZF9tYXA6IGRpY3QpIC0+IGdwZC5HZW9EYXRhRnJhbWU6CiAgICBpZiAiZl90eXBlIiBub3QgaW4gZ2RmLmNvbHVtbnM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJ7bGF5ZXJ9IHBhcnF1ZXQgaGFzIG5vIGZfdHlwZSBjb2x1bW46IGNvbHVtbnM9e2xpc3QoZ2RmLmNvbHVtbnMpfSIKICAgICAgICApCiAgICBraW5kcyA9IGdkZlsiZl90eXBlIl0ubWFwKGtpbmRfbWFwKQogICAgdW5rbm93biA9IHNvcnRlZChzZXQoZ2RmWyJmX3R5cGUiXS51bmlxdWUoKSkgLSBzZXQoa2luZF9tYXApKQogICAgaWYgdW5rbm93bjoKICAgICAgICBwcmludChmIklORk86IGV4Y2x1ZGVkIGZfdHlwZSB2YWx1ZXM6IHt1bmtub3dufSIsIGZpbGU9c3lzLnN0ZGVycikKICAgIGtlZXAgPSBraW5kcy5ub3RuYSgpCiAgICBnZGYgPSBnZGZba2VlcF0uY29weSgpCiAgICBnZGZbImtpbmQiXSA9IGtpbmRzW2tlZXBdLnZhbHVlcwogICAgcmV0dXJuIGdkZgoKCmRlZiBfY29udHJhY3RfZ2RmKGdkZjogZ3BkLkdlb0RhdGFGcmFtZSkgLT4gZ3BkLkdlb0RhdGFGcmFtZToKICAgIHJldHVybiBncGQuR2VvRGF0YUZyYW1lKAogICAgICAgIHsic291cmNlIjogIm1sIiwgImtpbmQiOiBnZGZbImtpbmQiXS52YWx1ZXMsICJjb25maWRlbmNlIjogW05vbmVdICogbGVuKGdkZil9LAogICAgICAgIGdlb21ldHJ5PWdkZi5nZW9tZXRyeS52YWx1ZXMsCiAgICAgICAgY3JzPTQzMjYsCiAgICApCgoKZGVmIF9tZXRyaWMoZ2RmOiBncGQuR2VvRGF0YUZyYW1lLCBsb246IGZsb2F0KSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgem9uZSA9IGludCgobG9uICsgMTgwKSAvLyA2KSArIDMyNjAxCiAgICByZXR1cm4gZ2RmLnRvX2Nycyh6b25lKQoKCmRlZiBjb2xsYXBzZV9jcm9zc3dhbGtzKGxpbmVzOiBncGQuR2VvRGF0YUZyYW1lLCB0aHJlc2hvbGQ6IGZsb2F0ID0gMS41KSAtPiBncGQuR2VvRGF0YUZyYW1lOgogICAgIiIiQ2x1c3RlciBuZWFyLWR1cGxpY2F0ZSBwYXJhbGxlbCBjcm9zc3dhbGsgY2VudGVybGluZXMgd2l0aGluCiAgICBgdGhyZXNob2xkYCBtZXRlcnMgKG1ldHJpYykgYW5kIGtlZXAgdGhlIGxvbmdlc3QgbGluZSBwZXIgY2x1c3Rlci4KICAgIFJldHVybnMgdGhlIGZpbHRlcmVkIEdlb0RhdGFGcmFtZSAoYWxsIGtpbmRzKTsgY3Jvc3N3YWxrIGNvdW50IGJlZm9yZS9hZnRlcgogICAgaXMgcmVwb3J0ZWQgYnkgdGhlIGNhbGxlciB2aWEgbGVuKCkgZGlmZnMuCiAgICAiIiIKICAgIGltcG9ydCBtYXRoCgogICAgbG9uID0gKGxpbmVzLnRvdGFsX2JvdW5kc1swXSArIGxpbmVzLnRvdGFsX2JvdW5kc1syXSkgLyAyLjAKICAgIHpvbmUgPSBpbnQoKGxvbiArIDE4MCkgLy8gNikgKyAzMjYwMQogICAgbWV0cmljID0gbGluZXMudG9fY3JzKHpvbmUpCgogICAgZ2VvbXMgPSBtZXRyaWMuZ2VvbWV0cnkudmFsdWVzCiAgICBrZWVwID0gbnAub25lcyhsZW4oZ2VvbXMpLCBkdHlwZT1ib29sKQoKICAgIGlzX2N3ID0gbWV0cmljWyJraW5kIl0udmFsdWVzID09ICJjcm9zc3dhbGsiCiAgICBjd19pZHggPSBucC53aGVyZShpc19jdylbMF0KICAgIGlmIGxlbihjd19pZHgpIDw9IDE6CiAgICAgICAgcmV0dXJuIGxpbmVzCgogICAgYnVmZmVycyA9IFtnLmJ1ZmZlcih0aHJlc2hvbGQgLyAyLjApIGlmIGcgaXMgbm90IE5vbmUgZWxzZSBOb25lIGZvciBnIGluIGdlb21zXQogICAgdHJlZSA9IHNoYXBlbHkuU1RSdHJlZShbYiBmb3IgYiBpbiBidWZmZXJzIGlmIGIgaXMgbm90IE5vbmVdKQoKICAgICMgdW5pb24tZmluZCBvdmVyIGNyb3Nzd2FsayBsaW5lcyB3aG9zZSBidWZmZXJzIGludGVyc2VjdAogICAgcGFyZW50ID0gbGlzdChyYW5nZShsZW4oZ2VvbXMpKSkKCiAgICBkZWYgZmluZChpOiBpbnQpIC0+IGludDoKICAgICAgICB3aGlsZSBwYXJlbnRbaV0gIT0gaToKICAgICAgICAgICAgcGFyZW50W2ldID0gcGFyZW50W3BhcmVudFtpXV0KICAgICAgICAgICAgaSA9IHBhcmVudFtpXQogICAgICAgIHJldHVybiBpCgogICAgZm9yIGkgaW4gY3dfaWR4OgogICAgICAgIGlmIGJ1ZmZlcnNbaV0gaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3IgaiBpbiB0cmVlLnF1ZXJ5KGJ1ZmZlcnNbaV0pOgogICAgICAgICAgICBqID0gaW50KGopCiAgICAgICAgICAgIGlmIGkgPj0gaiBvciBub3QgaXNfY3dbal0gb3IgYnVmZmVyc1tqXSBpcyBOb25lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgYnVmZmVyc1tpXS5pbnRlcnNlY3RzKGJ1ZmZlcnNbal0pOgogICAgICAgICAgICAgICAgcmksIHJqID0gZmluZChpKSwgZmluZChqKQogICAgICAgICAgICAgICAgaWYgcmkgIT0gcmo6CiAgICAgICAgICAgICAgICAgICAgcGFyZW50W3JqXSA9IHJpCgogICAgY2x1c3RlcnM6IGRpY3RbaW50LCBsaXN0W2ludF1dID0ge30KICAgIGZvciBpIGluIGN3X2lkeDoKICAgICAgICBjbHVzdGVycy5zZXRkZWZhdWx0KGZpbmQoaW50KGkpKSwgW10pLmFwcGVuZChpbnQoaSkpCgogICAgZHJvcHBlZCA9IDAKICAgIGZvciBtZW1iZXJzIGluIGNsdXN0ZXJzLnZhbHVlcygpOgogICAgICAgIGlmIGxlbihtZW1iZXJzKSA8PSAxOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG1lbWJlcl9sZW5ndGhzID0gW2dlb21zW21dLmxlbmd0aCBpZiBnZW9tc1ttXSBpcyBub3QgTm9uZSBlbHNlIDAgZm9yIG0gaW4gbWVtYmVyc10KICAgICAgICBiZXN0ID0gbWVtYmVyc1tpbnQobnAuYXJnbWF4KG1lbWJlcl9sZW5ndGhzKSldCiAgICAgICAgZm9yIG0gaW4gbWVtYmVyczoKICAgICAgICAgICAgaWYgbSAhPSBiZXN0OgogICAgICAgICAgICAgICAga2VlcFttXSA9IEZhbHNlCiAgICAgICAgICAgICAgICBkcm9wcGVkICs9IDEKCiAgICBpZiBkcm9wcGVkOgogICAgICAgIHByaW50KGYiSU5GTzogY29sbGFwc2VkIHtkcm9wcGVkfSBuZWFyLWR1cGxpY2F0ZSBjcm9zc3dhbGsgbGluZXMgIgogICAgICAgICAgICAgIGYiKHtsZW4oY3dfaWR4KX0gLT4ge2xlbihjd19pZHgpIC0gZHJvcHBlZH0gaW4gY2x1c3RlcnMpIiwgZmlsZT1zeXMuc3RkZXJyKQogICAgcmV0dXJuIGxpbmVzW2tlZXBdLCBkcm9wcGVkCgoKZGVmIGNvbnZlcnRfcHJvamVjdCgKICAgICAgICBwcm9qZWN0X2Rpcjogc3RyIHwgcGF0aGxpYi5QYXRoLAogICAgICAgIHByb2plY3RfbmFtZTogc3RyLAogICAgICAgIGluZmVycmVkX291dDogc3RyIHwgcGF0aGxpYi5QYXRoLAogICAgICAgIHBvbHlnb25zX291dDogc3RyIHwgcGF0aGxpYi5QYXRoLAogICAgICAgICAgICAgICAgY29sbGFwc2VfY3c6IGJvb2wgPSBGYWxzZSwKKSAtPiB0dXBsZVtpbnQsIGludCwgaW50XToKICAgICIiIlJldHVybnMgKG5fbGluZXNfd3JpdHRlbiwgbl9wb2x5Z29uc193cml0dGVuLCBuX2Ryb3BwZWRfZHVwbGljYXRlcykuIiIiCiAgICBwcm9qZWN0X2RpciA9IHBhdGhsaWIuUGF0aChwcm9qZWN0X2RpcikKICAgIG5ldF9wYXRoID0gcHJvamVjdF9kaXIgLyAibmV0d29yayIgLyBmIntwcm9qZWN0X25hbWV9LW5ldHdvcmsucGFycXVldCIKICAgIHBvbF9wYXRoID0gcHJvamVjdF9kaXIgLyAicG9seWdvbnMiIC8gZiJ7cHJvamVjdF9uYW1lfS1wb2x5Z29ucy5wYXJxdWV0IgoKICAgIGlmIG5vdCBuZXRfcGF0aC5leGlzdHMoKSBhbmQgbm90IHBvbF9wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIk5laXRoZXIge25ldF9wYXRofSBub3Ige3BvbF9wYXRofSBmb3VuZDsgIgogICAgICAgICAgICBmImNoZWNrIC0tcHJvamVjdC1kaXIvLS1wcm9qZWN0LW5hbWUuIgogICAgICAgICkKCiAgICBuX3BvbHlzID0gbl9kcm9wcGVkID0gMAoKICAgIGlmIG5ldF9wYXRoLmV4aXN0cygpOgogICAgICAgIG5ldCA9IF9tYXBfa2luZHMoX3RvXzQzMjYoZ3BkLnJlYWRfcGFycXVldChuZXRfcGF0aCkpLCAibmV0d29yayIsIExJTkVfS0lORFMpCiAgICAgICAgbGluZXMgPSBfY29udHJhY3RfZ2RmKG5ldCkKICAgICAgICBpZiBsZW4obGluZXMpOgogICAgICAgICAgICBsaW5lcyA9IGxpbmVzLmV4cGxvZGUoaW5kZXhfcGFydHM9RmFsc2UpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICAgICAgbGluZXMgPSBsaW5lc1tsaW5lcy5nZW9tZXRyeS5nZW9tX3R5cGUgPT0gIkxpbmVTdHJpbmciXQogICAgICAgICAgICBpZiBjb2xsYXBzZV9jdzoKICAgICAgICAgICAgICAgIGxpbmVzLCBkcm9wcGVkX2N3ID0gY29sbGFwc2VfY3Jvc3N3YWxrcyhsaW5lcykKICAgICAgICAgICAgICAgIG5fZHJvcHBlZCA9IGRyb3BwZWRfY3cKICAgICAgICBpZiBsZW4obGluZXMpOgogICAgICAgICAgICBpbmZlcnJlZF9vdXQgPSBwYXRobGliLlBhdGgoaW5mZXJyZWRfb3V0KQogICAgICAgICAgICBpbmZlcnJlZF9vdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgbGluZXMudG9fZmlsZShpbmZlcnJlZF9vdXQsIGRyaXZlcj0iR2VvSlNPTiIpCiAgICAgICAgICAgIG5fbGluZXMgPSBsZW4obGluZXMpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJpbnQoIldBUk5JTkc6IG5vIGxpbmUgZmVhdHVyZXMgaW4gbmV0d29yayBwYXJxdWV0IiwgZmlsZT1zeXMuc3RkZXJyKQogICAgICAgICAgICBuX2xpbmVzID0gMAoKICAgIGlmIHBvbF9wYXRoLmV4aXN0cygpOgogICAgICAgIHBvbCA9IF9tYXBfa2luZHMoX3RvXzQzMjYoZ3BkLnJlYWRfcGFycXVldChwb2xfcGF0aCkpLCAicG9seWdvbnMiLCBQT0xZR09OX0tJTkRTKQogICAgICAgIHBvbHlzID0gX2NvbnRyYWN0X2dkZihwb2wpCiAgICAgICAgaWYgbGVuKHBvbHlzKToKICAgICAgICAgICAgcG9seWdvbnNfb3V0ID0gcGF0aGxpYi5QYXRoKHBvbHlnb25zX291dCkKICAgICAgICAgICAgcG9seWdvbnNfb3V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIHBvbHlzLnRvX2ZpbGUocG9seWdvbnNfb3V0LCBkcml2ZXI9Ikdlb0pTT04iKQogICAgICAgICAgICBuX3BvbHlzID0gbGVuKHBvbHlzKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KCJXQVJOSU5HOiBubyBwb2x5Z29uIGZlYXR1cmVzIGluIHBvbHlnb25zIHBhcnF1ZXQiLCBmaWxlPXN5cy5zdGRlcnIpCgogICAgcmV0dXJuIG5fbGluZXMsIG5fcG9seXMsIG5fZHJvcHBlZAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigKICAgICAgICBkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyCiAgICApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1kaXIiLCByZXF1aXJlZD1UcnVlLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InRpbGUybmV0IG91dHB1dCBkaXIgZm9yIG9uZSBwcm9qZWN0IChjb250YWlucyBuZXR3b3JrLyBhbmQgcG9seWdvbnMvKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1uYW1lIiwgcmVxdWlyZWQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJwcm9qZWN0IG5hbWUgdXNlZCB3aXRoIHRpbGUybmV0IGdlbmVyYXRlIC1uIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pbmZlcnJlZCIsIHJlcXVpcmVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ib3V0cHV0IEdlb0pTT04gcGF0aCBmb3IgdGhlIGNlbnRlcmxpbmUgbmV0d29yayIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcG9seWdvbnMiLCByZXF1aXJlZD1UcnVlLAogICAgICAgICAgICAgICAgICAgIGhlbHA9Im91dHB1dCBHZW9KU09OIHBhdGggZm9yIHRoZSBwb2x5Z29ucyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29sbGFwc2UtY3Jvc3N3YWxrcyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iY29sbGFwc2UgbmVhci1kdXBsaWNhdGUgcGFyYWxsZWwgY3Jvc3N3YWxrIGNlbnRlcmxpbmVzICIKICAgICAgICAgICAgICAgICAgICAgICAgICIoY2x1c3RlciB3aXRoaW4gfjEuNSBtLCBrZWVwIHRoZSBsb25nZXN0IG9mIGVhY2ggY2x1c3RlcikiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIG5fbGluZXMsIG5fcG9seXMsIG5fZHJvcHBlZCA9IGNvbnZlcnRfcHJvamVjdCgKICAgICAgICBhcmdzLnByb2plY3RfZGlyLCBhcmdzLnByb2plY3RfbmFtZSwgYXJncy5pbmZlcnJlZCwgYXJncy5wb2x5Z29ucywKICAgICAgICBjb2xsYXBzZV9jdz1hcmdzLmNvbGxhcHNlX2Nyb3Nzd2Fsa3MsCiAgICApCiAgICBwcmludChmIndyb3RlIHtuX2xpbmVzfSBsaW5lIGZlYXR1cmVzIC0+IHthcmdzLmluZmVycmVkfSIpCiAgICBwcmludChmIndyb3RlIHtuX3BvbHlzfSBwb2x5Z29uIGZlYXR1cmVzIC0+IHthcmdzLnBvbHlnb25zfSIpCiAgICBpZiBhcmdzLmNvbGxhcHNlX2Nyb3Nzd2Fsa3M6CiAgICAgICAgcHJpbnQoZiJjb2xsYXBzZWQge25fZHJvcHBlZH0gZHVwbGljYXRlIGNyb3Nzd2FsayBsaW5lcyIpCiAgICBpZiBuX2xpbmVzID09IDAgYW5kIG5fcG9seXMgPT0gMDoKICAgICAgICBwcmludCgiV0FSTklORzogYm90aCBvdXRwdXRzIGVtcHR5OyBjaGVjayB0aWxlMm5ldCBpbmZlcmVuY2UgbG9ncyIsIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICByZXR1cm4gMQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==",
    "osm_metrics.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJDb21wYXJlIHRpbGUybmV0IG91dHB1dCBhZ2FpbnN0IE9wZW5TdHJlZXRNYXAgZm9yIGEgZGVtbyBiYm94IChDUFUpLgoKTWV0cmljcyBjb21wdXRlZCAodjIgZGVzaWduKToKICAxLiBTZXBhcmF0ZSBjb3VudHMvbGVuZ3RocyBmb3IgKGEpIGZvb3R3YXk9c2lkZXdhbGsgd2F5cyBhbmQgKGIpIHVudHlwZWQKICAgICBoaWdod2F5PWZvb3R3YXkgd2F5cyDigJQgTk9UIG1lcmdlZCAoZGlmZmVyZW50IHNlbWFudGljIHF1YWxpdHkpLgogIDIuIFJlY2FsbCBwcm94aWVzICh3aXRoaW4gMyBtIG9mIHRpbGUybmV0IGNlbnRlcmxpbmVzKSBjb21wdXRlZAogICAgIHNlcGFyYXRlbHkgZm9yIHRhZ2dlZCBzaWRld2Fsa3MsIHVudHlwZWQgZm9vdHdheXMsIGFuZCBjcm9zc2luZ3MuCiAgMy4gUm9hZC1zaWRld2FsayByZWNhbGw6IHJvYWRzIHRhZ2dlZCBzaWRld2Fsaz1sZWZ0L3JpZ2h0L2JvdGgveWVzIOKAlCBzaGFyZQogICAgIG9mIHN1Y2ggcm9hZCBsZW5ndGggd2l0aCBhIHRpbGUybmV0IHNpZGV3YWxrIGxpbmUgd2l0aGluIDE1IG0uCiAgNC4gImNhbmRpZGF0ZV9uZXdfZWRnZXMiIOKAlCB0aWxlMm5ldCBzaWRld2Fsay9jcm9zc3dhbGsgbGVuZ3RoIHdpdGggbm8gT1NNCiAgICAgc2lkZXdhbGstb3ItZm9vdHdheSB3aXRoaW4gNSBtLiBDQU5ESURBVEUtT05MWSBsYWJlbDogaW5jbHVkZXMgZmFsc2UKICAgICBwb3NpdGl2ZXMgKGd1dHRlcnMsIHBhcmtpbmcgYWlzbGVzLCBwcml2YXRlIGRyaXZlcyk7IHZlcmlmeSBiZWZvcmUgdXNlLgoKQ2F2ZWF0OiBPcGVuU3RyZWV0TWFwIHNpZGV3YWxrIGRhdGEgaXMgaW5jb21wbGV0ZSBhbmQgaXMgbm90IGdyb3VuZCB0cnV0aDsKdGhlc2UgYXJlIHByb3h5IG1ldHJpY3MsIG5vdCBhY2N1cmFjeSBtZWFzdXJlbWVudHMuCgpVc2FnZToKICBweXRob24gb3NtX21ldHJpY3MucHkgLS1pbmZlcnJlZCBkYXRhL2RlbW9fYXJlYV9pbmZlcnJlZC5nZW9qc29uIFwKICAgICAgLS1iYm94ICIzOC45MTgyLC03Ny4wMzgwLDM4LjkwOTIsLTc3LjAyNzAiIFwKICAgICAgLS1yZWdpb24gd2FzaGluZ3Rvbi1kYyAtLW91dCBtZXRyaWNzL2FlcmlhbF92c19vc20uanNvbiBcCiAgICAgIFstLWR1bXAtbmV3LWVkZ2VzIG1sL2FlcmlhbC9xYS9uZXdfZWRnZXNfYWxsLmdlb2pzb25dCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGRhdGV0aW1lCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCBwYXRobGliCmltcG9ydCBzeXMKaW1wb3J0IHdhcm5pbmdzCgppbXBvcnQgZ2VvcGFuZGFzIGFzIGdwZAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IG9zbW54IGFzIG94CmltcG9ydCBzaGFwZWx5Cgp3aXRoIHdhcm5pbmdzLmNhdGNoX3dhcm5pbmdzKCk6CiAgICB3YXJuaW5ncy5zaW1wbGVmaWx0ZXIoImlnbm9yZSIpCgpXQUxLX0tJTkRTID0geyJzaWRld2FsayIsICJmb290cGF0aCIsICJjcm9zc3dhbGsiLCAiY29ubmVjdG9yIn0KClRSQUlOSU5HX1JFR0lPTlMgPSB7CiAgICAid2FzaGluZ3Rvbi1kYyI6ICJEQyIsCiAgICAiYm9zdG9uIjogIkJvc3RvbiwgTUEiLAogICAgImNhbWJyaWRnZSI6ICJDYW1icmlkZ2UsIE1BIiwKICAgICJueWMiOiAiTmV3IFlvcmssIE5ZIiwKICAgICJuZXcteW9yay1jaXR5IjogIk5ldyBZb3JrLCBOWSIsCiAgICAibWFuaGF0dGFuIjogIk5ldyBZb3JrLCBOWSIsCn0KClJPQURfSElHSFdBWVMgPSBbInJlc2lkZW50aWFsIiwgInNlY29uZGFyeSIsICJwcmltYXJ5IiwgInRlcnRpYXJ5IiwKICAgICAgICAgICAgICAgICAidW5jbGFzc2lmaWVkIiwgImxpdmluZ19zdHJlZXQiLCAic2VydmljZSIsICJzZWNvbmRhcnlfbGluayIsCiAgICAgICAgICAgICAgICAgInByaW1hcnlfbGluayIsICJ0ZXJ0aWFyeV9saW5rIl0KV0VTVCwgQk9UVE9NLCBFQVNULCBUT1AgPSAwLCAxLCAyLCAzCgoKZGVmIF91dG1fZXBzZyhsb246IGZsb2F0KSAtPiBpbnQ6CiAgICB6b25lID0gbWF0aC5mbG9vcigobG9uICsgMTgwKSAvIDYpICsgMQogICAgcmV0dXJuICgzMjYwMCArIHpvbmUpIGlmIGxvbiA+PSAwIGVsc2UgKDMyNzAwICsgem9uZSkKCgpkZWYgX2Zvb3R3YXlfd2F5cyhiYm94KToKICAgIGh3ID0gb3guZmVhdHVyZXNfZnJvbV9iYm94KGJib3g9YmJveCwgdGFncz17ImhpZ2h3YXkiOiBbImZvb3R3YXkiXX0pCiAgICBodyA9IGh3LnJlc2V0X2luZGV4KCkKICAgIGh3ID0gaHdbKGh3LmdldCgiZWxlbWVudF90eXBlIiwgaHcuZ2V0KCJlbGVtZW50IikpID09ICJ3YXkiKV0KICAgIGh3ID0gaHdbaHcuZ2VvbWV0cnkuZ2VvbV90eXBlLmlzaW4oWyJMaW5lU3RyaW5nIiwgIk11bHRpTGluZVN0cmluZyJdKV0KICAgIGYgPSBod1siZm9vdHdheSJdLmFzdHlwZShzdHIpIGlmICJmb290d2F5IiBpbiBody5jb2x1bW5zIGVsc2UgIm5hbiIKCiAgICB0YWdnZWQgPSBod1soZiA9PSAic2lkZXdhbGsiKV0uY29weSgpCiAgICB0YWdnZWRbIm9zbV9ncm91cCJdID0gImZvb3R3YXlfc2lkZXdhbGtfdGFnZ2VkIgoKICAgIHVudHlwZWQgPSBod1soZiA9PSAibmFuIildLmNvcHkoKQogICAgdW50eXBlZFsib3NtX2dyb3VwIl0gPSAiZm9vdHdheV91bnR5cGVkIgoKICAgIGNyb3NzaW5nID0gaHdbKGYgPT0gImNyb3NzaW5nIildLmNvcHkoKQogICAgY3Jvc3NpbmdbIm9zbV9ncm91cCJdID0gImZvb3R3YXlfY3Jvc3NpbmciCgogICAgcmV0dXJuIGdwZC5HZW9EYXRhRnJhbWUoCiAgICAgICAgZ3BkLkdlb0RhdGFGcmFtZShwZF9jb25jYXRfc2FmZShbdGFnZ2VkLCB1bnR5cGVkLCBjcm9zc2luZ10pLCBjcnM9aHcuY3JzKSwKICAgICAgICBjcnM9aHcuY3JzLAogICAgKQoKCmRlZiBwZF9jb25jYXRfc2FmZShmcmFtZXMpOgogICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgcmV0dXJuIHBkLmNvbmNhdChmcmFtZXMsIGlnbm9yZV9pbmRleD1UcnVlKQoKCmRlZiBfc2lkZXdhbGtfcm9hZHMoYmJveCk6CiAgICByb2FkcyA9IG94LmZlYXR1cmVzX2Zyb21fYmJveCgKICAgICAgICBiYm94PWJib3gsCiAgICAgICAgdGFncz17ImhpZ2h3YXkiOiBST0FEX0hJR0hXQVlTLCAic2lkZXdhbGsiOiBbImxlZnQiLCAicmlnaHQiLCAiYm90aCIsICJ5ZXMiXX0sCiAgICApCiAgICByb2FkcyA9IHJvYWRzLnJlc2V0X2luZGV4KCkKICAgIGVsZW0gPSAiZWxlbWVudF90eXBlIiBpZiAiZWxlbWVudF90eXBlIiBpbiByb2Fkcy5jb2x1bW5zIGVsc2UgImVsZW1lbnQiCiAgICByb2FkcyA9IHJvYWRzW3JvYWRzW2VsZW1dID09ICJ3YXkiXQogICAgcm9hZHMgPSByb2Fkc1tyb2Fkcy5nZW9tZXRyeS5nZW9tX3R5cGUuaXNpbihbIkxpbmVTdHJpbmciLCAiTXVsdGlMaW5lU3RyaW5nIl0pXQogICAgcmV0dXJuIGdwZC5HZW9EYXRhRnJhbWUocm9hZHNbWyJnZW9tZXRyeSJdXSwgY3JzPXJvYWRzLmNycykKCgpkZWYgdG9fbWV0cmljKGdkZiwgbG9uKToKICAgIHJldHVybiBnZGYudG9fY3JzKF91dG1fZXBzZyhsb24pKQoKCmRlZiBldmFsdWF0ZShpbmZlcnJlZF9wYXRoLCBiYm94LCByZWdpb246IHN0cikgLT4gdHVwbGVbZGljdCwgZ3BkLkdlb0RhdGFGcmFtZSB8IE5vbmVdOgogICAgbGF0X3RvcCwgbG9uX2xlZnQsIGxhdF9ib3R0b20sIGxvbl9yaWdodCA9IGJib3gKICAgIG9zbV9iYm94ID0gKGxvbl9sZWZ0LCBsYXRfYm90dG9tLCBsb25fcmlnaHQsIGxhdF90b3ApCiAgICBjZW50ZXJfbG9uID0gKGxvbl9sZWZ0ICsgbG9uX3JpZ2h0KSAvIDIuMAoKICAgIGluZmVycmVkID0gZ3BkLnJlYWRfZmlsZShpbmZlcnJlZF9wYXRoKQogICAgaWYgaW5mZXJyZWQuY3JzIGlzIE5vbmU6CiAgICAgICAgaW5mZXJyZWQgPSBpbmZlcnJlZC5zZXRfY3JzKDQzMjYpCiAgICBlbGlmIGluZmVycmVkLmNycy50b19lcHNnKCkgIT0gNDMyNjoKICAgICAgICBpbmZlcnJlZCA9IGluZmVycmVkLnRvX2Nycyg0MzI2KQoKICAgIFQyTiA9IHRvX21ldHJpYyhpbmZlcnJlZCwgY2VudGVyX2xvbikKICAgIHQybl93YWxrID0gVDJOW1QyTlsia2luZCJdLmlzaW4oV0FMS19LSU5EUyldCiAgICB0Mm5fc2lkZXMgPSBUMk5bVDJOWyJraW5kIl0gPT0gInNpZGV3YWxrIl0KICAgIHQybl9zaWRlc191bmlvbiA9ICgKICAgICAgICBzaGFwZWx5LnVuaW9uX2FsbCh0Mm5fc2lkZXMuZ2VvbWV0cnkudmFsdWVzKSBpZiBsZW4odDJuX3NpZGVzKSBlbHNlIE5vbmUKICAgICkKCiAgICBmb290ID0gdG9fbWV0cmljKF9mb290d2F5X3dheXMob3NtX2Jib3gpLCBjZW50ZXJfbG9uKQogICAgZ3JvdXBzID0gbGlzdChmb290WyJvc21fZ3JvdXAiXS51bmlxdWUoKSkKCiAgICBvdXQ6IGRpY3QgPSB7Im9zbV9ncm91cHMiOiB7fX0KICAgIGZvciBncnAgaW4gWyJmb290d2F5X3NpZGV3YWxrX3RhZ2dlZCIsICJmb290d2F5X3VudHlwZWQiLCAiZm9vdHdheV9jcm9zc2luZyJdOgogICAgICAgIHN1YiA9IGZvb3RbZm9vdFsib3NtX2dyb3VwIl0gPT0gZ3JwXQogICAgICAgIHJlYyA9IE5vbmUKICAgICAgICBsZW5ndGggPSBmbG9hdChzdWIuZ2VvbWV0cnkubGVuZ3RoLnN1bSgpKSBpZiBsZW4oc3ViKSBlbHNlIDAuMAogICAgICAgIGlmIHN1Yi5lbXB0eToKICAgICAgICAgICAgb3V0WyJvc21fZ3JvdXBzIl1bZ3JwXSA9IHsKICAgICAgICAgICAgICAgICJjb3VudCI6IDAsICJsZW5ndGhfbSI6IDAuMCwKICAgICAgICAgICAgICAgICJyZWNhbGxfd2l0aGluXzNtX3NoYXJlIjogTm9uZSwKICAgICAgICAgICAgfQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIHQybl93YWxrIGlzIG5vdCBOb25lOgogICAgICAgICAgICB1ID0gc2hhcGVseS51bmlvbl9hbGwoc3ViLmdlb21ldHJ5LnZhbHVlcykKICAgICAgICAgICAgd2l0aGluID0gc2hhcGVseS5pbnRlcnNlY3Rpb24oCiAgICAgICAgICAgICAgICB1LCBzaGFwZWx5LnVuaW9uX2FsbCh0Mm5fd2Fsay5nZW9tZXRyeS52YWx1ZXMpLmJ1ZmZlcigzLjApCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmVjID0gKCBzaGFwZWx5Lmxlbmd0aCh3aXRoaW4pIC8gbGVuZ3RoKSBpZiBsZW5ndGggZWxzZSBOb25lCiAgICAgICAgb3V0WyJvc21fZ3JvdXBzIl1bZ3JwXSA9IHsKICAgICAgICAgICAgImNvdW50IjogaW50KGxlbihzdWIpKSwKICAgICAgICAgICAgImxlbmd0aF9tIjogcm91bmQobGVuZ3RoLCAxKSwKICAgICAgICAgICAgInJlY2FsbF93aXRoaW5fM21fc2hhcmUiOiByb3VuZChmbG9hdChyZWMpLCA0KSBpZiByZWMgaXMgbm90IE5vbmUgZWxzZSBOb25lLAogICAgICAgIH0KCiAgICAjIC0tLSByb2FkLXNpZGV3YWxrIHJlY2FsbCAoQDE1IG0gZnJvbSB0aWxlMm5ldCBzaWRld2FsayBsaW5lcykKICAgIHJvYWRzID0gdG9fbWV0cmljKF9zaWRld2Fsa19yb2Fkcyhvc21fYmJveCksIGNlbnRlcl9sb24pCiAgICBpZiBsZW4ocm9hZHMpOgogICAgICAgIHJvYWRfbGVuID0gZmxvYXQocm9hZHMuZ2VvbWV0cnkubGVuZ3RoLnN1bSgpKQogICAgICAgIHJvYWRfbGVuX2hpdCA9IDAuMAogICAgICAgIGlmIHQybl9zaWRlc191bmlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgY2x1dGNoID0gc2hhcGVseS51bmlvbl9hbGwocm9hZHMuZ2VvbWV0cnkudmFsdWVzKQogICAgICAgICAgICBoaXQgPSBzaGFwZWx5LmludGVyc2VjdGlvbihjbHV0Y2gsIHQybl9zaWRlc191bmlvbi5idWZmZXIoMTUuMCkpCiAgICAgICAgICAgIHJvYWRfbGVuX2hpdCA9IHNoYXBlbHkubGVuZ3RoKGhpdCkKICAgICAgICByb2FkX3JlY2FsbCA9IHJvYWRfbGVuX2hpdCAvIHJvYWRfbGVuIGlmIHJvYWRfbGVuIGVsc2UgTm9uZQogICAgZWxzZToKICAgICAgICByb2FkX2xlbiwgcm9hZF9sZW5faGl0LCByb2FkX3JlY2FsbCA9IDAuMCwgMC4wLCBOb25lCiAgICBvdXRbInJvYWRzX3NpZGV3YWxrX3RhZ2dlZCJdID0gewogICAgICAgICJjb3VudCI6IGludChsZW4ocm9hZHMpKSwKICAgICAgICAibGVuZ3RoX20iOiByb3VuZChyb2FkX2xlbiwgMSksCiAgICAgICAgImxlbmd0aF93aXRoX3Qybl9zaWRld2Fsa193aXRoaW5fMTVtX20iOiByb3VuZChyb2FkX2xlbl9oaXQsIDEpLAogICAgICAgICJzaWRld2Fsa19yZWNhbGxfc2hhcmUiOiByb3VuZChmbG9hdChyb2FkX3JlY2FsbCksIDQpIGlmIHJvYWRfcmVjYWxsIGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgIH0KCiAgICAjIC0tLSBjYW5kaWRhdGUgbmV3IGVkZ2VzIChDQU5ESURBVEUtT05MWSkgKyBvcHRpb25hbCBkdW1wCiAgICBvc21fd2Fsa191bmlvbiA9ICgKICAgICAgICBzaGFwZWx5LnVuaW9uX2FsbChmb290Lmdlb21ldHJ5LnZhbHVlcykgaWYgbGVuKGZvb3QpIGVsc2UgTm9uZQogICAgKQogICAgdDJuX3VuaW9uID0gKAogICAgICAgIHNoYXBlbHkudW5pb25fYWxsKHQybl93YWxrLmdlb21ldHJ5LnZhbHVlcykgaWYgbGVuKHQybl93YWxrKSBlbHNlIE5vbmUKICAgICkKICAgIG5ld19lZGdlc19nZGYgPSBOb25lCiAgICBpZiB0Mm5fdW5pb24gaXMgbm90IE5vbmU6CiAgICAgICAgYmV5b25kID0gdDJuX3VuaW9uIGlmIG9zbV93YWxrX3VuaW9uIGlzIE5vbmUgZWxzZSBzaGFwZWx5LmRpZmZlcmVuY2UoCiAgICAgICAgICAgIHQybl91bmlvbiwgb3NtX3dhbGtfdW5pb24uYnVmZmVyKDUuMCkKICAgICAgICApCiAgICAgICAgYmV5b25kX2xlbiA9IGZsb2F0KHNoYXBlbHkubGVuZ3RoKGJleW9uZCkpCiAgICAgICAgdDJuX2xlbiA9IGZsb2F0KHQybl93YWxrLmdlb21ldHJ5Lmxlbmd0aC5zdW0oKSkKICAgICAgICBvdXRbImNhbmRpZGF0ZV9uZXdfZWRnZXNfQ0FORElEQVRFX09OTFkiXSA9IHsKICAgICAgICAgICAgImRlZmluaXRpb24iOiAidGlsZTJuZXQgc2lkZXdhbGsvY3Jvc3N3YWxrL2Nvbm5lY3RvciBsZW5ndGggd2l0aCBubyBPU00gIgogICAgICAgICAgICAgICAgICAgICAgICAgICJzaWRld2Fsay1vci1mb290d2F5IHdpdGhpbiA1IG07IGluY2x1ZGVzIGZhbHNlIHBvc2l0aXZlcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgIihndXR0ZXJzLCBwYXJraW5nIGFpc2xlcywgcHJpdmF0ZSBkcml2ZXMpIGFuZCBtdXN0IGJlICIKICAgICAgICAgICAgICAgICAgICAgICAgICAiaHVtYW4vUUEgdmVyaWZpZWQgYmVmb3JlIHJvdXRlciB1c2UiLAogICAgICAgICAgICAibGVuZ3RoX20iOiByb3VuZChiZXlvbmRfbGVuLCAxKSwKICAgICAgICAgICAgInNoYXJlX29mX3RpbGUybmV0IjogKAogICAgICAgICAgICAgICAgcm91bmQoYmV5b25kX2xlbiAvIHQybl9sZW4sIDQpIGlmIHQybl9sZW4gZWxzZSBOb25lCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJ0aWxlMm5ldF90b3RhbF9sZW5ndGhfbSI6IHJvdW5kKHQybl9sZW4sIDEpLAogICAgICAgIH0KICAgICAgICBpZiBsZW4oaW5mZXJyZWQpOgogICAgICAgICAgICBtYXNrID0gW10KICAgICAgICAgICAgb3NtX2J1ZiA9IG9zbV93YWxrX3VuaW9uLmJ1ZmZlcig1LjApIGlmIG9zbV93YWxrX3VuaW9uIGlzIG5vdCBOb25lIGVsc2UgTm9uZQogICAgICAgICAgICBmb3IgZ2VvbSBpbiB0Mm5fd2Fsay5nZW9tZXRyeS52YWx1ZXM6CiAgICAgICAgICAgICAgICBpZiBvc21fYnVmIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgbWFzay5hcHBlbmQoVHJ1ZSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgaW50ZXIgPSBzaGFwZWx5LmludGVyc2VjdGlvbihnZW9tLCBvc21fYnVmKQogICAgICAgICAgICAgICAgICAgIG1hc2suYXBwZW5kKHNoYXBlbHkubGVuZ3RoKGludGVyKSA8PSAxZS05KQogICAgICAgICAgICBuZXdfZWRnZXNfZ2RmID0gdDJuX3dhbGtbbnAuYXJyYXkobWFzayldLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCiAgICBpbl90cmFpbmluZyA9IHJlZ2lvbi5sb3dlcigpLnN0cmlwKCkgaW4gVFJBSU5JTkdfUkVHSU9OUwogICAgcmVzdWx0ID0gewogICAgICAgICJnZW5lcmF0ZWRfdXRjIjogZGF0ZXRpbWUuZGF0ZXRpbWUubm93KGRhdGV0aW1lLnRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgInJlZ2lvbiI6IHJlZ2lvbiwKICAgICAgICAiZGVtb19iYm94IjogbGlzdChiYm94KSwKICAgICAgICAiaW5fdHJhaW5pbmdfY2l0aWVzIjogaW5fdHJhaW5pbmcsCiAgICAgICAgInRyYWluaW5nX2NpdGllcyI6IHNvcnRlZChzZXQoVFJBSU5JTkdfUkVHSU9OUy52YWx1ZXMoKSkpLAogICAgICAgICoqb3V0LAogICAgICAgICJjYXZlYXQiOiAoCiAgICAgICAgICAgICJPcGVuU3RyZWV0TWFwIHNpZGV3YWxrIGRhdGEgaXMgaW5jb21wbGV0ZSBhbmQgbm90IGdyb3VuZCB0cnV0aDsgIgogICAgICAgICAgICAidGhlc2UgYXJlIHByb3h5IG1ldHJpY3MsIG5vdCBhY2N1cmFjeS4gVGlsZTJOZXQgZGlkIG5vdCB0cmFpbiBvbiAiCiAgICAgICAgICAgICJ0aGlzIHJlZ2lvbiB1bmxlc3MgaW5fdHJhaW5pbmdfY2l0aWVzIGlzIHRydWUuICIKICAgICAgICAgICAgImNhbmRpZGF0ZV9uZXdfZWRnZXMgaXMgQ0FORElEQVRFLU9OTFkgKHVudmVyaWZpZWQpLiIKICAgICAgICApLAogICAgfQogICAgcmV0dXJuIHJlc3VsdCwgbmV3X2VkZ2VzX2dkZgoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigKICAgICAgICBkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyCiAgICApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taW5mZXJyZWQiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJib3giLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJsYXQsbG9uLGxhdCxsb24gKHRvcC1sZWZ0IHRoZW4gYm90dG9tLXJpZ2h0KSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVnaW9uIiwgZGVmYXVsdD0id2FzaGluZ3Rvbi1kYyIsCiAgICAgICAgICAgICAgICAgICAgaGVscD1mImtleSBmb3IgaW5fdHJhaW5pbmdfY2l0aWVzIGNoZWNrOyB1c2UgZS5nLiBhcmxpbmd0b24tdmEgZm9yIG5vbi10cmFpbmluZyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kdW1wLW5ldy1lZGdlcyIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJvcHRpb25hbCBHZW9KU09OIHBhdGggZm9yIHRoZSBpbmRpdmlkdWFsIGNhbmRpZGF0ZS1uZXctZWRnZSBsaW5lcyIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgYmJveCA9IHR1cGxlKGZsb2F0KHYpIGZvciB2IGluIGFyZ3MuYmJveC5zcGxpdCgiLCIpKQogICAgaWYgbGVuKGJib3gpICE9IDQ6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1iYm94IG11c3QgYmUgbGF0LGxvbixsYXQsbG9uIikKCiAgICB0cnk6CiAgICAgICAgKG1ldHJpY3MsIG5ld19lZGdlcykgPSBldmFsdWF0ZShhcmdzLmluZmVycmVkLCBiYm94LCBhcmdzLnJlZ2lvbikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIkVSUk9SOiB7ZX0iLCBmaWxlPXN5cy5zdGRlcnIpCiAgICAgICAgcmV0dXJuIDEKCiAgICBvdXQgPSBwYXRobGliLlBhdGgoYXJncy5vdXQpCiAgICBvdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dC53cml0ZV90ZXh0KGpzb24uZHVtcHMobWV0cmljcywgaW5kZW50PTIpICsgIlxuIikKCiAgICBpZiBhcmdzLmR1bXBfbmV3X2VkZ2VzIGFuZCBuZXdfZWRnZXMgaXMgbm90IE5vbmUgYW5kIGxlbihuZXdfZWRnZXMpOgogICAgICAgIHAgPSBwYXRobGliLlBhdGgoYXJncy5kdW1wX25ld19lZGdlcykKICAgICAgICBwLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgd2FudCA9IG5ld19lZGdlc1sobmV3X2VkZ2VzWyJraW5kIl0uaXNpbihbInNpZGV3YWxrIiwgImNyb3Nzd2FsayJdKSldLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICB3YW50NDMyNiA9IHdhbnQudG9fY3JzKDQzMjYpCiAgICAgICAgZyA9IGdwZC5HZW9EYXRhRnJhbWUoCiAgICAgICAgICAgIHsia2luZCI6IHdhbnQ0MzI2WyJraW5kIl0udmFsdWVzLCAic291cmNlIjogIm1sIiwKICAgICAgICAgICAgICJjb25maWRlbmNlIjogW05vbmVdKmxlbih3YW50NDMyNil9LAogICAgICAgICAgICBnZW9tZXRyeT13YW50NDMyNi5nZW9tZXRyeS52YWx1ZXMsIGNycz00MzI2LAogICAgICAgICkKICAgICAgICBnLnRvX2ZpbGUocCwgZHJpdmVyPSJHZW9KU09OIikKICAgICAgICBwcmludChmImR1bXBlZCB7bGVuKGcpfSBjYW5kaWRhdGUtbmV3LWVkZ2UgbGluZXMgLT4ge3B9IikKICAgIHByaW50KGpzb24uZHVtcHMobWV0cmljcywgaW5kZW50PTIpKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==",
    "make_previews.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWlsZCBwcmV2aWV3IG1hcHMgKGZvbGl1bSBIVE1MICsgbWF0cGxvdGxpYiBQTkcpIGZyb20gY29udHJhY3QgR2VvSlNPTnMKClVzYWdlOgogIHB5dGhvbiBtYWtlX3ByZXZpZXdzLnB5IC0taW5mZXJyZWQgZGF0YS94X2luZmVycmVkLmdlb2pzb24gXAogICAgICAtLXBvbHlnb25zIGRhdGEveF9wb2x5Z29ucy5nZW9qc29uIFwKICAgICAgLS1iYm94ICJsYXQsbG9uLGxhdCxsb24iIC0tbmFtZSB0ZXN0X2FyZWEgLS1vdXQgbWwvYWVyaWFsL3ByZXZpZXdzCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHBhdGhsaWIKaW1wb3J0IHdhcm5pbmdzCgppbXBvcnQgZm9saXVtCmltcG9ydCBnZW9wYW5kYXMgYXMgZ3BkCmltcG9ydCBtYXRwbG90bGliCgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgc2hhcGVseQoKI21hdGggcGxvdCB1c2FnZQoKS0lORF9DT0xPUlMgPSB7CiAgICAic2lkZXdhbGsiOiAiIzFmNzdiNCIsCiAgICAiY3Jvc3N3YWxrIjogIiNmZjdmMGUiLAogICAgImZvb3RwYXRoIjogIiMyY2EwMmMiLAp9CgoKZGVmIF9sb2FkKHBhdGg6IHN0ciB8IHBhdGhsaWIuUGF0aCwgY3JzPTQzMjYpIC0+IGdwZC5HZW9EYXRhRnJhbWU6CiAgICBnZGYgPSBncGQucmVhZF9maWxlKHBhdGgpCiAgICBpZiBnZGYuY3JzIGlzIE5vbmU6CiAgICAgICAgZ2RmID0gZ2RmLnNldF9jcnMoY3JzKQogICAgZWxpZiBnZGYuY3JzLnRvX2Vwc2coKSAhPSBjcnM6CiAgICAgICAgZ2RmID0gZ2RmLnRvX2NycyhjcnMpCiAgICByZXR1cm4gZ2RmCgoKZGVmIGJ1aWxkKAogICAgICAgIGluZmVycmVkX3BhdGg6IHN0ciwKICAgICAgICBwb2x5Z29uc19wYXRoOiBzdHIgfCBOb25lLAogICAgICAgIGJib3g6IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXQsIGZsb2F0XSwKICAgICAgICBuYW1lOiBzdHIsCiAgICAgICAgb3V0X2Rpcjogc3RyIHwgcGF0aGxpYi5QYXRoLAopIC0+IHR1cGxlW3BhdGhsaWIuUGF0aCwgcGF0aGxpYi5QYXRoXToKICAgIG91dF9kaXIgPSBwYXRobGliLlBhdGgob3V0X2RpcikKICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGxhdF90b3AsIGxvbl9sZWZ0LCBsYXRfYm90dG9tLCBsb25fcmlnaHQgPSBiYm94CiAgICBjZW50ZXIgPSAoKGxhdF90b3AgKyBsYXRfYm90dG9tKSAvIDIuMCwgKGxvbl9sZWZ0ICsgbG9uX3JpZ2h0KSAvIDIuMCkKCiAgICBmbWFwID0gZm9saXVtLk1hcChsb2NhdGlvbj1jZW50ZXIsIHpvb21fc3RhcnQ9MTYpCiAgICBiYm94X3BvbHkgPSBmb2xpdW0uUG9seWdvbigKICAgICAgICBsb2NhdGlvbnM9WyhsYXRfdG9wLCBsb25fbGVmdCksIChsYXRfdG9wLCBsb25fcmlnaHQpLAogICAgICAgICAgICAgICAgICAgKGxhdF9ib3R0b20sIGxvbl9yaWdodCksIChsYXRfYm90dG9tLCBsb25fbGVmdCldLAogICAgICAgIGNvbG9yPSJibGFjayIsIHdlaWdodD0yLCBmaWxsPUZhbHNlLCB0b29sdGlwPSJiYm94IiwKICAgICkKICAgIGJib3hfcG9seS5hZGRfdG8oZm1hcCkKCiAgICBraW5kX29mID0gbGFtYmRhIGZlYXR1cmU6ICgoZmVhdHVyZSBvciB7fSkuZ2V0KCJwcm9wZXJ0aWVzIikgb3Ige30pLmdldCgia2luZCIpICAjIG5vcWE6IEU3MzEKCiAgICBpZiBpbmZlcnJlZF9wYXRoIGFuZCBwYXRobGliLlBhdGgoaW5mZXJyZWRfcGF0aCkuZXhpc3RzKCk6CiAgICAgICAgbGluZXMgPSBfbG9hZChpbmZlcnJlZF9wYXRoKQogICAgICAgIHN0eWxlID0gbGFtYmRhIGZlYXR1cmU6IHsgICMgbm9xYTogRTczMQogICAgICAgICAgICAiY29sb3IiOiBLSU5EX0NPTE9SUy5nZXQoa2luZF9vZihmZWF0dXJlKSwgIiM4ODg4ODgiKSwKICAgICAgICAgICAgIndlaWdodCI6IDMsCiAgICAgICAgfQogICAgICAgIGZvbGl1bS5HZW9Kc29uKGxpbmVzLl9fZ2VvX2ludGVyZmFjZV9fLCBzdHlsZV9mdW5jdGlvbj1zdHlsZSwKICAgICAgICAgICAgICAgICAgICAgICBuYW1lPSJjZW50ZXJsaW5lcyIpLmFkZF90byhmbWFwKQogICAgaWYgcG9seWdvbnNfcGF0aCBhbmQgcGF0aGxpYi5QYXRoKHBvbHlnb25zX3BhdGgpLmV4aXN0cygpOgogICAgICAgIHBvbHlzID0gX2xvYWQocG9seWdvbnNfcGF0aCkKICAgICAgICBwc3R5bGUgPSBsYW1iZGEgZmVhdHVyZTogeyAgIyBub3FhOiBFNzMxCiAgICAgICAgICAgICJjb2xvciI6IEtJTkRfQ09MT1JTLmdldChraW5kX29mKGZlYXR1cmUpLCAiIzg4ODg4OCIpLAogICAgICAgICAgICAid2VpZ2h0IjogMSwgImZpbGxPcGFjaXR5IjogMC4yNSwKICAgICAgICB9CiAgICAgICAgZm9saXVtLkdlb0pzb24ocG9seXMuX19nZW9faW50ZXJmYWNlX18sIHN0eWxlX2Z1bmN0aW9uPXBzdHlsZSwKICAgICAgICAgICAgICAgICAgICAgICBuYW1lPSJwb2x5Z29ucyIpLmFkZF90byhmbWFwKQogICAgZm9saXVtLkxheWVyQ29udHJvbCgpLmFkZF90byhmbWFwKQogICAgZm1hcC5maXRfYm91bmRzKFtbbGF0X2JvdHRvbSwgbG9uX2xlZnRdLCBbbGF0X3RvcCwgbG9uX3JpZ2h0XV0pCgogICAgaHRtbF9wYXRoID0gb3V0X2RpciAvIGYie25hbWV9X3ByZXZpZXcuaHRtbCIKICAgIGZtYXAuc2F2ZShodG1sX3BhdGgpCgogICAgIyBtYXRwbG90bGliIFBORyAobm8gYmFzZW1hcCBkZXBlbmRlbmN5KQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4LCA4KSkKICAgIGJib3hfZ2RmID0gZ3BkLkdlb0RhdGFGcmFtZSgKICAgICAgICBnZW9tZXRyeT1bc2hhcGVseS5ib3gobG9uX2xlZnQsIGxhdF9ib3R0b20sIGxvbl9yaWdodCwgbGF0X3RvcCldLCBjcnM9NDMyNgogICAgKQogICAgYmJveF9nZGYuYm91bmRhcnkucGxvdChheD1heCwgY29sb3I9ImJsYWNrIiwgbGluZXdpZHRoPTEpCiAgICBpZiBwb2x5Z29uc19wYXRoIGFuZCBwYXRobGliLlBhdGgocG9seWdvbnNfcGF0aCkuZXhpc3RzKCk6CiAgICAgICAgcG9seXMgPSBfbG9hZChwb2x5Z29uc19wYXRoKQogICAgICAgIGZvciBraW5kLCBzdWIgaW4gcG9seXMuZ3JvdXBieSgia2luZCIpOgogICAgICAgICAgICBzdWIuYm91bmRhcnkucGxvdChheD1heCwgY29sb3I9S0lORF9DT0xPUlMuZ2V0KGtpbmQsICIjODg4IiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxpbmV3aWR0aD0wLjQsIGxhYmVsPWYie2tpbmR9IChwb2x5KSIpCiAgICBpZiBpbmZlcnJlZF9wYXRoIGFuZCBwYXRobGliLlBhdGgoaW5mZXJyZWRfcGF0aCkuZXhpc3RzKCk6CiAgICAgICAgbGluZXMgPSBfbG9hZChpbmZlcnJlZF9wYXRoKQogICAgICAgIGZvciBraW5kLCBzdWIgaW4gbGluZXMuZ3JvdXBieSgia2luZCIpOgogICAgICAgICAgICBzdWIucGxvdChheD1heCwgY29sb3I9S0lORF9DT0xPUlMuZ2V0KGtpbmQsICIjODg4IiksIGxpbmV3aWR0aD0xLjIsCiAgICAgICAgICAgICAgICAgICAgIGxhYmVsPWYie2tpbmR9IChsaW5lKSIpCiAgICBheC5sZWdlbmQobG9jPSJsb3dlciByaWdodCIsIGZvbnRzaXplPTcpCiAgICBheC5zZXRfYXhpc19vZmYoKQogICAgcG5nX3BhdGggPSBvdXRfZGlyIC8gZiJ7bmFtZX1fcHJldmlldy5wbmciCiAgICBmaWcuc2F2ZWZpZyhwbmdfcGF0aCwgZHBpPTE1MCwgYmJveF9pbmNoZXM9InRpZ2h0IikKICAgIHBsdC5jbG9zZShmaWcpCgogICAgcmV0dXJuIGh0bWxfcGF0aCwgcG5nX3BhdGgKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHdpdGggd2FybmluZ3MuY2F0Y2hfd2FybmluZ3MoKToKICAgICAgICB3YXJuaW5ncy5zaW1wbGVmaWx0ZXIoImlnbm9yZSIpCiAgICAgICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigKICAgICAgICAgICAgZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcgogICAgICAgICkKICAgICAgICBhcC5hZGRfYXJndW1lbnQoIi0taW5mZXJyZWQiLCByZXF1aXJlZD1UcnVlKQogICAgICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wb2x5Z29ucyIsIGRlZmF1bHQ9Tm9uZSkKICAgICAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmJveCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9ImxhdCxsb24sbGF0LGxvbiIpCiAgICAgICAgYXAuYWRkX2FyZ3VtZW50KCItLW5hbWUiLCByZXF1aXJlZD1UcnVlKQogICAgICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCByZXF1aXJlZD1UcnVlKQogICAgICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICAgICAgYmJveCA9IHR1cGxlKGZsb2F0KHYpIGZvciB2IGluIGFyZ3MuYmJveC5zcGxpdCgiLCIpKQogICAgICAgIGlmIGxlbihiYm94KSAhPSA0OgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCItLWJib3ggbXVzdCBiZSBsYXQsbG9uLGxhdCxsb24iKQogICAgICAgIGh0bWxfcGF0aCwgcG5nX3BhdGggPSBidWlsZCgKICAgICAgICAgICAgYXJncy5pbmZlcnJlZCwgYXJncy5wb2x5Z29ucywgYmJveCwgYXJncy5uYW1lLCBhcmdzLm91dAogICAgICAgICkKICAgICAgICBwcmludChmIndyb3RlIHtodG1sX3BhdGh9XG53cm90ZSB7cG5nX3BhdGh9IikKICAgICAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK",
}
for name, b in script_b64.items():
    pathlib.Path(BASE, name).write_bytes(base64.b64decode(b))
sys.path.insert(0, BASE)
import convert_tile2net_output
import make_previews
import osm_metrics
print("scripts written to", BASE)


In [ ]:

test_project = f"{OUTPUT_ROOT}/test_area"
demo_project = f"{OUTPUT_ROOT}/demo"

convert_tile2net_output.convert_project(
    test_project, "test_area",
    f"{DATA_DIR}/test_area_inferred.geojson",
    f"{DATA_DIR}/test_area_polygons.geojson",
)
convert_tile2net_output.convert_project(
    demo_project, "demo",
    f"{DATA_DIR}/demo_area_inferred.geojson",
    f"{DATA_DIR}/demo_area_polygons.geojson",
)
print("\ncontract files in", DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    print(" ", f)



## 6. Preview maps (folium HTML + PNG)


In [ ]:

make_previews.build(
    inferred_path=f"{DATA_DIR}/test_area_inferred.geojson",
    polygons_path=f"{DATA_DIR}/test_area_polygons.geojson",
    bbox=TEST_BBOX, name="test_area", out_dir=PREVIEW_DIR,
)
make_previews.build(
    inferred_path=f"{DATA_DIR}/demo_area_inferred.geojson",
    polygons_path=f"{DATA_DIR}/demo_area_polygons.geojson",
    bbox=DEMO_BBOX, name="demo_area", out_dir=PREVIEW_DIR,
)
print(os.listdir(PREVIEW_DIR))



## 7. OSM comparison metrics for the demo area

(a) share of OSM sidewalk/crossing length within 3 m of tile2net centerlines
(recall proxy); (b) share of tile2net sidewalk/crosswalk length with **no**
OSM sidewalk/footway within 5 m (candidate new edges); (c)
`in_training_cities` flag. OSM is incomplete and not ground truth — read the
caveat in the output JSON.


In [ ]:

import pathlib

result, new_edges_gdf = osm_metrics.evaluate(
    f"{DATA_DIR}/demo_area_inferred.geojson", DEMO_BBOX, DEMO_REGION
)
print(json.dumps(result, indent=2))
pathlib.Path(f"{METRICS_DIR}/aerial_vs_osm.json").write_text(json.dumps(result, indent=2) + "\n")
print("candidate new-edge lines (not exported to payload):", 0 if new_edges_gdf is None else len(new_edges_gdf))



## 8. Payload: parquet debug copies + zip download


In [ ]:

import shutil

for area, name in (("test_area", "test_area"), ("demo", "demo")):
    net_src = f"{OUTPUT_ROOT}/{area}/network/{name}-network.parquet"
    pol_src = f"{OUTPUT_ROOT}/{area}/polygons/{name}-polygons.parquet"
    for src in (net_src, pol_src):
        if os.path.exists(src):
            shutil.copy(src, f"{PQ_DIR}/{name}_{os.path.basename(src)}")
print(os.listdir(PQ_DIR))


In [ ]:

payload = f"{BASE}/tile2net_payload.zip"
p = sp.run(
    ["zip", "-r", payload, "data", "ml_aerial_previews", "metrics", "parquet_debug"],
    cwd=BASE, capture_output=True, text=True,
)
print(p.stdout[-2000:], p.stderr[-2000:])
print("payload:", payload, os.path.getsize(payload), "bytes")

try:
    from google.colab import files
    files.download(payload)
except Exception as e:
    print("auto-download unavailable:", e)
    print("Download tile2net_payload.zip from the file browser sidebar instead.")
